# Stage 3: Voice Recording PreprocessingThis notebook audits the group's real voice recordings, visualizes audio, applies controlled augmentation, and extracts speaker/audio features for later voiceprint verification.## Required recordingsCollect at least two genuine recordings per participant, with consent. Each participant should record both phrases:- **Yes, approve**- **Confirm transaction**Use WAV files (preferably mono PCM, 16 kHz or 44.1 kHz). Speak naturally in a quiet room, keep the microphone distance similar, and avoid sharing recordings publicly.## Folder layout```textdata/audio/raw/  Ella/    yes_approve/*.wav    confirm_transaction/*.wav  King/    yes_approve/*.wav    confirm_transaction/*.wav  Nicole/    yes_approve/*.wav    confirm_transaction/*.wav  Sarah/    yes_approve/*.wav    confirm_transaction/*.wav```The notebook also recognizes participant IDs in folder names or filenames. Phrase labels must be identifiable from a folder name or filename. It does not generate or invent recordings.

## 1. Imports and project pathsThis pipeline uses NumPy, pandas, Matplotlib, and SciPy, avoiding an extra audio package dependency. It processes WAV files and resamples them to 16 kHz mono for consistent feature extraction.

In [ ]:
import sysimport refrom pathlib import Pathfrom math import gcdimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom scipy.io import wavfilefrom scipy.signal import resample_poly, stft, istftfrom scipy.fft import dctfrom tqdm.auto import tqdmprint("Python:", sys.version.split()[0])print("NumPy:", np.__version__)print("Pandas:", pd.__version__)ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()RAW_DIR = ROOT / "data" / "audio" / "raw"PROCESSED_DIR = ROOT / "data" / "audio" / "processed"AUGMENTED_DIR = PROCESSED_DIR / "augmented"PROCESSED_DIR.mkdir(parents=True, exist_ok=True)AUGMENTED_DIR.mkdir(parents=True, exist_ok=True)TARGET_SR = 16000PARTICIPANTS = {    "ella": "participant_01", "participant_01": "participant_01",    "king": "participant_02", "participant_02": "participant_02",    "nicole": "participant_03", "participant_03": "participant_03",    "sarah": "participant_04", "participant_04": "participant_04",}PHRASES = {    "yes_approve": ["yes_approve", "yes approve", "approve"],    "confirm_transaction": ["confirm_transaction", "confirm transaction", "confirm"],}AUDIO_EXTENSIONS = {".wav"}print("Project root:", ROOT)print("Raw audio folder:", RAW_DIR)print("Raw folder exists:", RAW_DIR.exists())

## 2. Discover recordings and identify labelsEach recording must map to one participant and one phrase. Files with unclear labels are reported instead of being guessed.

In [ ]:
def path_tokens(path):    rel = path.relative_to(RAW_DIR)    return [part.lower().replace("-", "_").replace(" ", "_") for part in rel.parts[:-1]] + [        path.stem.lower().replace("-", "_").replace(" ", "_")    ]def identify_participant(path):    joined = "_".join(path_tokens(path))    for alias, label in PARTICIPANTS.items():        if re.search(rf"(?<![a-z0-9]){re.escape(alias)}(?![a-z0-9])", joined):            return label    return Nonedef identify_phrase(path):    joined = "_".join(path_tokens(path))    matches = []    for phrase, aliases in PHRASES.items():        if any(re.search(rf"(?<![a-z0-9]){re.escape(alias.replace(' ', '_'))}(?![a-z0-9])", joined)               for alias in aliases):            matches.append(phrase)    return matches[0] if len(matches) == 1 else Noneif not RAW_DIR.exists():    raise FileNotFoundError(        f"Audio folder not found: {RAW_DIR}\n"        "Create data/audio/raw/ and place the real participant WAV recordings inside it."    )audio_paths = sorted(p for p in RAW_DIR.rglob("*")                     if p.is_file() and p.suffix.lower() in AUDIO_EXTENSIONS)records, unresolved = [], []for path in audio_paths:    participant = identify_participant(path)    phrase = identify_phrase(path)    if participant is None or phrase is None:        unresolved.append({            "file": str(path.relative_to(RAW_DIR)),            "participant_detected": participant,            "phrase_detected": phrase        })    else:        records.append({"path": path, "participant_id": participant, "phrase": phrase})inventory = pd.DataFrame(records)print("WAV files found:", len(audio_paths))print("Recordings with recognized labels:", len(inventory))print("Unresolved files:", len(unresolved))display(inventory.head(20))if unresolved:    display(pd.DataFrame(unresolved))

## 3. Check recording countsThe table should show recordings for every participant and both phrases. Aim for at least two recordings per participant overall, ideally multiple takes of each phrase. More genuine samples are better than creating many augmented copies.

In [ ]:
if inventory.empty:    raise ValueError("No labeled WAV recordings found. Check the folder layout and filenames.")count_table = (inventory.groupby(["participant_id", "phrase"])               .size().unstack(fill_value=0)               .reindex(index=sorted(set(PARTICIPANTS.values())),                        columns=sorted(PHRASES), fill_value=0))display(count_table)participant_totals = inventory.groupby("participant_id").size().reindex(    sorted(set(PARTICIPANTS.values())), fill_value=0)print("Recordings per participant:")display(participant_totals.rename("recording_count").to_frame())if (participant_totals < 2).any():    print("WARNING: At least one participant has fewer than two original recordings.")if (count_table == 0).any().any():    print("WARNING: At least one participant/phrase combination is missing.")

## 4. Load, validate, and standardize audioRecordings are converted to mono floating-point audio and resampled to 16 kHz. The original files remain untouched. Very short, silent, or unreadable files are skipped and logged.

In [ ]:
def load_audio(path, target_sr=TARGET_SR):    sr, raw_audio = wavfile.read(str(path))    if np.issubdtype(raw_audio.dtype, np.integer):        info = np.iinfo(raw_audio.dtype)        audio = raw_audio.astype(np.float64) / max(abs(info.min), info.max)    else:        audio = raw_audio.astype(np.float64)    if audio.ndim == 2:        audio = audio.mean(axis=1)    audio = np.nan_to_num(audio)    if sr != target_sr:        common = gcd(int(sr), int(target_sr))        audio = resample_poly(audio, target_sr // common, sr // common)    peak = np.max(np.abs(audio)) if len(audio) else 0    if peak > 1:        audio = audio / peak    return target_sr, audio.astype(np.float32)audio_records, skipped = [], []for row in tqdm(inventory.itertuples(), total=len(inventory)):    try:        sr, signal = load_audio(row.path)        duration = len(signal) / sr        rms = float(np.sqrt(np.mean(signal ** 2))) if len(signal) else 0.0        if duration < 0.3:            raise ValueError(f"Recording too short ({duration:.2f} seconds)")        if rms < 1e-5:            raise ValueError("Recording is silent or nearly silent")        audio_records.append({            "source_path": str(row.path),            "participant_id": row.participant_id,            "phrase": row.phrase,            "sample_rate": sr,            "duration_seconds": duration,            "signal": signal        })    except Exception as exc:        skipped.append({"file": str(row.path), "reason": str(exc)})print("Successfully loaded recordings:", len(audio_records))print("Skipped recordings:", len(skipped))if skipped:    display(pd.DataFrame(skipped))

## 5. Visualize waveform and spectrogramInspect representative recordings. The waveform shows amplitude over time; the spectrogram shows how frequency content changes during speech.

In [ ]:
if not audio_records:    raise ValueError("No usable recordings were loaded. Check the skipped-recording report.")sample = audio_records[0]signal, sr = sample["signal"], sample["sample_rate"]time_axis = np.arange(len(signal)) / srfig, axes = plt.subplots(2, 1, figsize=(12, 6))axes[0].plot(time_axis, signal, linewidth=0.7)axes[0].set(title=f"Waveform: {sample['participant_id']} / {sample['phrase']}",            xlabel="Time (seconds)", ylabel="Amplitude")frequencies, times, spectrum = stft(signal, fs=sr, nperseg=512, noverlap=384)axes[1].pcolormesh(times, frequencies, 20 * np.log10(np.abs(spectrum) + 1e-8),                   shading="auto")axes[1].set_ylim(0, 8000)axes[1].set(title="Spectrogram", xlabel="Time (seconds)", ylabel="Frequency (Hz)")plt.tight_layout()plt.show()

## 6. Audio augmentationFor each original recording, create modest variations:- small pitch shifts (±2 semitones),- time stretching (0.9× and 1.1×) using a phase-vocoder approach,- low-level additive noise.Augmented recordings are training variants, not new independent voice samples. Keep every variant from one source recording in the same train/test split.

In [ ]:
def pitch_shift(signal, semitones):    factor = 2 ** (semitones / 12)    intermediate = max(16, int(len(signal) / factor))    shifted = np.interp(        np.linspace(0, len(signal) - 1, intermediate),        np.arange(len(signal)), signal    )    restored = np.interp(        np.linspace(0, len(shifted) - 1, len(signal)),        np.arange(len(shifted)), shifted    )    return restored.astype(np.float32)def time_stretch(signal, rate, n_fft=1024, hop=256):    if rate <= 0:        raise ValueError("rate must be positive")    _, _, z = stft(signal, fs=TARGET_SR, nperseg=n_fft,                   noverlap=n_fft-hop, boundary="zeros", padded=True)    steps = np.arange(0, z.shape[1] - 1, rate)    phase_advance = 2 * np.pi * hop * np.arange(n_fft // 2 + 1) / n_fft    output = np.zeros((z.shape[0], len(steps)), dtype=np.complex128)    phase = np.angle(z[:, 0])    for i, step in enumerate(steps):        left = int(step)        frac = step - left        magnitude = (1 - frac) * np.abs(z[:, left]) + frac * np.abs(z[:, left + 1])        delta = np.angle(z[:, left + 1]) - np.angle(z[:, left]) - phase_advance        delta = (delta + np.pi) % (2 * np.pi) - np.pi        phase += phase_advance + delta        output[:, i] = magnitude * np.exp(1j * phase)    _, stretched = istft(output, fs=TARGET_SR, nperseg=n_fft,                         noverlap=n_fft-hop, input_onesided=True,                         boundary=True)    target_len = max(1, int(len(signal) / rate))    if len(stretched) < target_len:        stretched = np.pad(stretched, (0, target_len - len(stretched)))    return stretched[:target_len].astype(np.float32)def add_noise(signal, snr_db=25, seed=42):    rng = np.random.default_rng(seed)    signal_power = np.mean(signal ** 2) + 1e-12    noise_power = signal_power / (10 ** (snr_db / 10))    noisy = signal + rng.normal(0, np.sqrt(noise_power), len(signal))    return np.clip(noisy, -1, 1).astype(np.float32)augmented_records = []for idx, item in enumerate(tqdm(audio_records)):    base = item["signal"]    variants = [        ("original", base),        ("pitch_up_2", pitch_shift(base, 2)),        ("pitch_down_2", pitch_shift(base, -2)),        ("time_stretch_0_9", time_stretch(base, 0.9)),        ("time_stretch_1_1", time_stretch(base, 1.1)),        ("noise_25db", add_noise(base, 25, seed=idx))    ]    for aug_name, audio in variants:        filename = f"{item['participant_id']}_{idx:04d}_{item['phrase']}_{aug_name}.wav"        out_path = AUGMENTED_DIR / filename        wavfile.write(str(out_path), TARGET_SR, audio)        augmented_records.append({            "audio_path": str(out_path),            "source_path": item["source_path"],            "participant_id": item["participant_id"],            "phrase": item["phrase"],            "augmentation": aug_name,            "signal": audio        })print("Original recordings:", len(audio_records))print("Augmented recordings including originals:", len(augmented_records))

## 7. Extract MFCC, spectral roll-off, and energy featuresThe feature extractor calculates 13 MFCC coefficients and summarizes their mean and standard deviation across frames. It also summarizes 85% spectral roll-off and frame-level RMS energy. These are compact baseline features for later voiceprint experiments, not a modern speaker embedding.

In [ ]:
def mel_to_hz(mel):    return 700 * (10 ** (mel / 2595) - 1)def hz_to_mel(hz):    return 2595 * np.log10(1 + hz / 700)def mel_filterbank(sr, n_fft=512, n_mels=26):    low_mel, high_mel = hz_to_mel(0), hz_to_mel(sr / 2)    mel_points = np.linspace(low_mel, high_mel, n_mels + 2)    hz_points = mel_to_hz(mel_points)    bins = np.floor((n_fft + 1) * hz_points / sr).astype(int)    filters = np.zeros((n_mels, n_fft // 2 + 1))    for m in range(1, n_mels + 1):        left, center, right = bins[m-1:m+2]        center = max(center, left + 1)        right = max(right, center + 1)        for k in range(left, min(center, filters.shape[1])):            filters[m-1, k] = (k-left) / (center-left)        for k in range(center, min(right, filters.shape[1])):            filters[m-1, k] = (right-k) / (right-center)    return filtersMEL_FILTERS = mel_filterbank(TARGET_SR)def extract_audio_features(signal, sr=TARGET_SR):    _, _, z = stft(signal, fs=sr, nperseg=512, noverlap=352,                   boundary=None, padded=False)    power = np.abs(z) ** 2    mel_power = MEL_FILTERS @ power    log_mel = np.log(mel_power + 1e-10)    mfcc = dct(log_mel, type=2, axis=0, norm="ortho")[:13]    freqs = np.linspace(0, sr / 2, power.shape[0])    cumulative = np.cumsum(power, axis=0)    totals = cumulative[-1, :] + 1e-12    rolloff_idx = np.argmax(cumulative >= 0.85 * totals[None, :], axis=0)    rolloff = freqs[rolloff_idx]    frame_energy = np.sqrt(np.mean(np.abs(z) ** 2, axis=0))    features = {}    for i in range(13):        features[f"mfcc_{i+1:02d}_mean"] = float(np.mean(mfcc[i]))        features[f"mfcc_{i+1:02d}_std"] = float(np.std(mfcc[i]))    features["spectral_rolloff_mean"] = float(np.mean(rolloff))    features["spectral_rolloff_std"] = float(np.std(rolloff))    features["rms_energy_mean"] = float(np.mean(frame_energy))    features["rms_energy_std"] = float(np.std(frame_energy))    features["duration_seconds"] = float(len(signal) / sr)    return featuresfeature_rows = []for item in tqdm(augmented_records):    row = {        "audio_path": item["audio_path"],        "source_path": item["source_path"],        "participant_id": item["participant_id"],        "phrase": item["phrase"],        "augmentation": item["augmentation"]    }    row.update(extract_audio_features(item["signal"]))    feature_rows.append(row)audio_features = pd.DataFrame(feature_rows)print("Audio feature table shape:", audio_features.shape)display(audio_features.head())

## 8. Save outputs and final validationThe main deliverable is `audio_features.csv`. The inventory and skipped-file report preserve traceability. Do not split augmented rows randomly: group by `source_path` or participant when designing later experiments.

In [ ]:
features_path = PROCESSED_DIR / "audio_features.csv"inventory_path = PROCESSED_DIR / "audio_inventory.csv"skipped_path = PROCESSED_DIR / "skipped_audio.csv"audio_features.to_csv(features_path, index=False)inventory.drop(columns=["path"]).assign(    path=inventory["path"].astype(str)).to_csv(inventory_path, index=False)pd.DataFrame(skipped, columns=["file", "reason"]).to_csv(skipped_path, index=False)print("Saved:")print(features_path)print(inventory_path)print(skipped_path)print("\nOriginal recordings represented by participant:")print(pd.DataFrame(audio_records).groupby("participant_id").size())print("\nFeature rows by augmentation:")display(audio_features["augmentation"].value_counts().to_frame("rows"))

## 9. Interpretation and limitations- Confirm all four participants and both phrases appear in the original recording inventory.- Listen to recordings and inspect waveforms/spectrograms for clipping, silence, background noise, and inconsistent microphone distance.- A large number of augmented rows does not replace collecting more genuine voice samples.- MFCC, spectral roll-off, and RMS energy are baseline handcrafted features. A later voice-verification stage should compare suitable classifiers and report false acceptance/false rejection behavior.- Split by original `source_path` or participant so augmented versions of the same recording cannot leak across train and test sets.- Voice recordings are biometric data. Obtain informed consent, restrict access, and do not commit raw recordings to a public repository.- The next stage is model development and the multimodal transaction simulation.